#<font color="Green">**Notebook Purpose**</font>

This notebook constructs a revised Table 1 (Patient Characteristics) for the DOM T2D manuscript. The key revision addresses Reviewer 1's Comment 1 by redefining baseline HbA1c and BMI as the closest recorded value **before** each patient's first GLM prescription date, rather than the first recorded value anywhere in the 2019–2024 study period.

**What changed from the original Table 1:**
- "First HbA1c in the study period" → "Baseline HbA1c (pre-treatment, 2019)"
- "First BMI in the study period" → "Baseline BMI (pre-treatment, 2019)"
- A "Missing" row is now reported for both HbA1c and BMI
- All other rows (demographics, comorbidities) are unchanged

**Rationale:** The original Table 1 presented the first recorded HbA1c for each patient with no temporal restriction. Since 40.9% of patients had their first HbA1c recorded *after* their first GLM prescription (median lag: 98 days), many "baseline" values actually reflected post-treatment levels. Among the 1,303 eligible patients (14%) with first HbA1c < 6.4%, 59.9% (780) had their first HbA1c after GLM initiation. This revision anchors clinical characteristics to a true pre-treatment window.

---

###<font color="Red"> Required Data </font>

To run the code blocks in this notebook, you will need the following **cleaned** CSVs (output from `CohortDatasetCreation.ipynb`):

1. **`medication_info.csv`** — columns: `patient_id`, `start_date`, `ingredient`, `medication_class`
2. **`lab_results.csv`** — columns: `patient_id`, `date`, `lab_result_num_val` (HbA1c, cleaned to 3.5–20% range)
3. **`BMI_vital_signs.csv`** — columns: `patient_id`, `date`, `value` (BMI)
4. **`patient_demographics.csv`** — columns: `patient_id`, `sex`, `race/ethnicity`, `year_of_birth`, `month_year_death`, `patient_regional_location`
5. **`patient_comorbidities.csv`** — columns: `patient_id`, `date`, `HF`, `CKD`
6. **`early_dropout_patients.pkl`** — dict of `{cluster_id: [patient_ids]}`

Additionally, after running **Section 2**, you must take the exported `patient_index.csv` to the `get_closest_continuous_value_after_t0` notebook to extract baseline values. You will then return with:

7. **`baseline_a1c.csv`** — output from `get_closest_continuous_value_after_t0`, containing `patient_id`, `baseline_a1c_value`, `baseline_a1c_days_from_t0`
8. **`baseline_bmi.csv`** — output from `get_closest_continuous_value_after_t0`, containing `patient_id`, `baseline_bmi_value`, `baseline_bmi_days_from_t0`

## Section 1 — Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import pickle as pkl

In [ ]:
medication_info = pd.read_csv('/content/medication_info.csv')
lab_results = pd.read_csv('/content/lab_results.csv')
BMI_vital_signs = pd.read_csv('/content/BMI_vital_signs.csv')
patient_demographics = pd.read_csv('/content/patient_demographics.csv')
patient_comorbidities = pd.read_csv('/content/patient_comorbidities.csv')

with open('/content/early_dropout_patients.pkl', 'rb') as f:
    early_dropout_patients = pkl.load(f)

In [ ]:
# Parse dates
medication_info['start_date'] = pd.to_datetime(medication_info['start_date'], errors='coerce')
lab_results['date'] = pd.to_datetime(lab_results['date'], errors='coerce')
BMI_vital_signs['date'] = pd.to_datetime(BMI_vital_signs['date'], errors='coerce')

## Section 2 — Build Patient Index (First GLM Date per Patient)

We compute each patient's first GLM prescription date. This becomes the **t0** anchor for baseline extraction. The output is a one-row-per-patient dataframe that will be exported and used as `index_df` in the `get_closest_continuous_value_after_t0` notebook.

In [ ]:
patient_index = (
    medication_info
    .sort_values('start_date')
    .groupby('patient_id', as_index=False)['start_date']
    .first()
    .rename(columns={'start_date': 't0'})
)

print(f"Patient index created: {len(patient_index):,} patients")
print(f"First GLM date range: {patient_index['t0'].min()} to {patient_index['t0'].max()}")
patient_index.head()

### <font color="Red">EXPORT: `patient_index.csv`</font>

Export this file and take it to the `get_closest_continuous_value_after_t0` notebook.

**For baseline HbA1c extraction**, use these parameters:
- `index_df` = `patient_index.csv`
- `values_df` = `lab_results.csv`, **pre-filtered to 2019** (`date >= '2019-01-01'` and `date < '2020-01-01'`)
- `target_days_from_t0 = 0`
- `days_before_target_date = 183`
- `days_after_target_date = 0`
- `value_name = 'baseline_a1c'`
- `value_col = 'lab_result_num_val'`
- `value_date_col = 'date'`
- `prefer_value = 'min'`

**For baseline BMI extraction**, use the same parameters except:
- `values_df` = `BMI_vital_signs.csv`, **pre-filtered to 2019**
- `value_name = 'baseline_bmi'`
- `value_col = 'value'`

Export the results as `baseline_a1c.csv` and `baseline_bmi.csv`, then return to **Section 3** below.

In [ ]:
patient_index.to_csv('/content/patient_index.csv', index=False)
print("Exported patient_index.csv")

---

## <font color="Orange">BREAK POINT</font>

**Go to `get_closest_continuous_value_after_t0` notebook now.**

1. Load `patient_index.csv` as your `index_df`
2. Pre-filter `lab_results` to 2019 dates only before passing as `values_df`
3. Run extraction with the parameters above for both HbA1c and BMI
4. Export as `baseline_a1c.csv` and `baseline_bmi.csv`
5. Return here and continue with Section 3

---

## Section 3 — Load Baseline Extraction Results

Load the baseline HbA1c and BMI CSVs produced by `get_closest_continuous_value_after_t0`.

In [ ]:
baseline_a1c = pd.read_csv('/content/baseline_a1c.csv')
baseline_bmi = pd.read_csv('/content/baseline_bmi.csv')

print(f"Baseline HbA1c records loaded: {len(baseline_a1c):,}")
print(f"  - with a value: {baseline_a1c['baseline_a1c_value'].notna().sum():,}")
print(f"  - missing: {baseline_a1c['baseline_a1c_value'].isna().sum():,}")
print()
print(f"Baseline BMI records loaded: {len(baseline_bmi):,}")
print(f"  - with a value: {baseline_bmi['baseline_bmi_value'].notna().sum():,}")
print(f"  - missing: {baseline_bmi['baseline_bmi_value'].isna().sum():,}")

## Section 4 — Exclude Early Dropouts

Filter all dataframes to the 9,327 eligible patients (those NOT in early dropout clusters).

In [ ]:
# Flatten early dropout patient IDs
dropout_ids = set()
for _, id_list in early_dropout_patients.items():
    dropout_ids.update([str(x) for x in id_list])

def exclude_dropouts(df, pid_col='patient_id'):
    out = df.copy()
    out[pid_col] = out[pid_col].astype(str)
    out = out[~out[pid_col].isin(dropout_ids)]
    return out

# Apply exclusion
dem = exclude_dropouts(patient_demographics)
a1c = exclude_dropouts(baseline_a1c)
bmi = exclude_dropouts(baseline_bmi)
com = exclude_dropouts(patient_comorbidities)

DENOMINATOR = len(dem)
print(f"Eligible patients (denominator): {DENOMINATOR:,}")

## Section 5 — Assemble Revised Table 1

In [ ]:
# =========================
# Configuration
# =========================
PCT_DECIMALS = 1

RACE_INPUT_LEVELS = ['White', 'Hispanic', 'Asian', 'Black', 'Other']
RACE_OUTPUT_MAP = {
    'Black':    'African American/Black, NH',
    'Asian':    'Asian, NH',
    'Hispanic': 'Hispanic/Latinx',
    'White':    'White, NH',
    'Other':    'Other, NH',
}

REGION_LEVELS = ['Midwest', 'South', 'West', 'Northeast']

AGE_BINS   = [-np.inf, 45, 55, 65, np.inf]
AGE_LABELS = ['- 18-45', '- 45-54', '- 55-64', '- 65+']

A1C_BINS   = [-np.inf, 6.4, 8.0, 9.0, np.inf]
A1C_LABELS = ['- <6.4', '- 6.5-7.9', '- 8.0-8.9', '- >= 9.0']

BMI_BINS   = [-np.inf, 24.9, 29.9, 34.9, 39.9, np.inf]
BMI_LABELS = ['- <24.9', '- 25.0-29.9', '- 30.0-34.9', '- 35.0-39.9', '- >= 40']

WINDOW_END = pd.Timestamp('2019-06-01')

def fmt(n):
    if pd.isna(n):
        n = 0
    n = int(n)
    pct = (n / DENOMINATOR) * 100 if DENOMINATOR else 0.0
    pct_str = f'{pct:.{PCT_DECIMALS}f}%'
    return f'n={n:,}, {pct_str}'

In [ ]:
# =========================
# 1) Demographics
# =========================

# Female sex
n_female = (dem['sex'].astype(str).str.upper().str.strip().isin(['F', 'FEMALE'])).sum()

# Age at baseline (2019)
dem['age_2019'] = 2019 - pd.to_numeric(dem['year_of_birth'], errors='coerce')
age_cuts = pd.cut(dem['age_2019'], bins=AGE_BINS,
                  labels=[l.replace('- ', '') for l in AGE_LABELS], right=False)
age_cnt = age_cuts.value_counts()
age_counts = {}
for edge_lbl, display_lbl in zip(age_cnt.index.categories, AGE_LABELS):
    age_counts[display_lbl] = int(age_cnt.get(edge_lbl, 0))

# Race/ethnicity
race_series = dem['race/ethnicity'].astype(str).str.strip()
race_counts = {}
for in_lbl in RACE_INPUT_LEVELS:
    out_lbl = RACE_OUTPUT_MAP[in_lbl]
    race_counts[f'- {out_lbl}'] = int((race_series == in_lbl).sum())

# US Region
region_series = dem['patient_regional_location'].astype(str).str.strip()
region_counts = {f'- {r}': int((region_series == r).sum()) for r in REGION_LEVELS}

In [ ]:
# =========================
# 2) Clinical Characteristics — REVISED baseline values
# =========================

# --- Baseline HbA1c (pre-treatment, 2019) ---
a1c_vals = pd.to_numeric(a1c['baseline_a1c_value'], errors='coerce')
a1c_valid = a1c_vals.dropna()
n_a1c_missing = int(a1c_vals.isna().sum())

a1c_binned = pd.cut(a1c_valid, bins=A1C_BINS,
                    labels=[l.replace('- ', '') for l in A1C_LABELS], right=False)
a1c_cnt = a1c_binned.value_counts()
a1c_counts = {}
for edge_lbl, display_lbl in zip(a1c_cnt.index.categories, A1C_LABELS):
    a1c_counts[display_lbl] = int(a1c_cnt.get(edge_lbl, 0))

# --- Baseline BMI (pre-treatment, 2019) ---
bmi_vals = pd.to_numeric(bmi['baseline_bmi_value'], errors='coerce')
bmi_valid = bmi_vals.dropna()
n_bmi_missing = int(bmi_vals.isna().sum())

bmi_binned = pd.cut(bmi_valid, bins=BMI_BINS,
                    labels=[l.replace('- ', '') for l in BMI_LABELS], right=True)
bmi_cnt = bmi_binned.value_counts()
bmi_counts = {}
for edge_lbl, display_lbl in zip(bmi_cnt.index.categories, BMI_LABELS):
    bmi_counts[display_lbl] = int(bmi_cnt.get(edge_lbl, 0))

In [ ]:
# =========================
# 3) Comorbidities (anytime up to June 1, 2019)
# =========================
com['date'] = pd.to_datetime(com['date'], errors='coerce')
window = com[com['date'] <= WINDOW_END].copy()

for col in ['HF', 'CKD']:
    window[col] = window[col].astype(str).str.strip().str.upper().isin(['TRUE', '1', 'T', 'Y', 'YES'])

hf_count  = int(window.loc[window['HF'],  'patient_id'].astype(str).nunique())
ckd_count = int(window.loc[window['CKD'], 'patient_id'].astype(str).nunique())

In [ ]:
# =========================
# 4) Build the display table
# =========================
rows = []
col_left  = 'Patient Characteristics'
col_right = f'Study Cohort, N = {DENOMINATOR:,}'

# ---- Demographics ----
rows.append(('Demographics', ''))
rows.append(('- Female sex', fmt(n_female)))

rows.append(('Age category, years', ''))
for label in AGE_LABELS:
    rows.append((label, fmt(age_counts.get(label, 0))))

rows.append(('Race/ethnicity', ''))
for out_label in ['- African American/Black, NH',
                  '- Asian, NH',
                  '- Hispanic/Latinx',
                  '- White, NH',
                  '- Other, NH']:
    rows.append((out_label, fmt(race_counts.get(out_label, 0))))

rows.append(('US Region', ''))
for r in [f'- {x}' for x in REGION_LEVELS]:
    rows.append((r, fmt(region_counts.get(r, 0))))

# ---- Clinical Characteristics (REVISED) ----
rows.append(('Clinical Characteristics', ''))

rows.append(('Baseline HbA1c (pre-treatment, 2019)', ''))
for label in A1C_LABELS:
    rows.append((label, fmt(a1c_counts.get(label, 0))))
rows.append(('- Missing', fmt(n_a1c_missing)))

rows.append(('Baseline BMI (pre-treatment, 2019)', ''))
for label in BMI_LABELS:
    rows.append((label, fmt(bmi_counts.get(label, 0))))
rows.append(('- Missing', fmt(n_bmi_missing)))

# ---- Comorbidities ----
rows.append(('Comorbidities before June 1, 2019', ''))
rows.append(('- Heart Failure', fmt(hf_count)))
rows.append(('- Chronic Kidney Disease', fmt(ckd_count)))

summary_df = pd.DataFrame(rows, columns=[col_left, col_right])
summary_df

## Section 6 — Comparison with Original Table 1

Side-by-side comparison of original vs. revised HbA1c and BMI distributions. Use these numbers in the response letter to quantify the impact of the redefinition.

In [ ]:
# Original Table 1 values (from manuscript Table 1)
original_a1c = {
    '<6.4':    1303,
    '6.5-7.9': 3967,
    '8.0-8.9': 1011,
    '>= 9.0':  2346,
    'Missing':    0,
}

original_bmi = {
    '<24.9':    355,
    '25.0-29.9': 1100,
    '30.0-34.9': 1621,
    '35.0-39.9': 1287,
    '>= 40':   1413,
    'Missing': DENOMINATOR - (355 + 1100 + 1621 + 1287 + 1413),
}

# Revised values
revised_a1c = {}
for label in A1C_LABELS:
    clean_label = label.replace('- ', '')
    revised_a1c[clean_label] = a1c_counts.get(label, 0)
revised_a1c['Missing'] = n_a1c_missing

revised_bmi = {}
for label in BMI_LABELS:
    clean_label = label.replace('- ', '')
    revised_bmi[clean_label] = bmi_counts.get(label, 0)
revised_bmi['Missing'] = n_bmi_missing

# Print comparison
print('=' * 60)
print('HbA1c COMPARISON: Original vs. Revised (Pre-treatment, 2019)')
print('=' * 60)
print(f'{"Category":<15} {"Original":>10} {"Revised":>10} {"Diff":>10}')
print('-' * 50)
for cat in original_a1c:
    orig = original_a1c[cat]
    rev = revised_a1c.get(cat, 0)
    print(f'{cat:<15} {orig:>10,} {rev:>10,} {rev - orig:>+10,}')

print()
print('=' * 60)
print('BMI COMPARISON: Original vs. Revised (Pre-treatment, 2019)')
print('=' * 60)
print(f'{"Category":<15} {"Original":>10} {"Revised":>10} {"Diff":>10}')
print('-' * 50)
for cat in original_bmi:
    orig = original_bmi[cat]
    rev = revised_bmi.get(cat, 0)
    print(f'{cat:<15} {orig:>10,} {rev:>10,} {rev - orig:>+10,}')

In [ ]:
# =========================
# 6b) Missing-baseline diagnostic check (standalone — no external function)
# =========================
# Two questions for the response letter:
#   (1) Of primary-cohort patients with no pre-GLM HbA1c in 2019,
#       how many had at least one HbA1c recorded ELSEWHERE in 2019?
#   (2) For those patients, how soon after GLM did their first HbA1c appear?
#
# Approach: existence check, not closest-value extraction. A patient
# "has baseline" if ANY HbA1c falls in [t0-183, t0] within 2019.
# This reproduces eTable 4's missing flag (should yield 3,810 missing).

# --- Primary cohort (exclude early dropouts) ---
dropout_ids = set()
for _, id_list in early_dropout_patients.items():
    dropout_ids.update([str(x) for x in id_list])

pi = patient_index.copy()
pi['patient_id'] = pi['patient_id'].astype(str)
primary_cohort = pi[~pi['patient_id'].isin(dropout_ids)].copy()

N_PRIMARY = len(primary_cohort)
print(f"Primary cohort: {N_PRIMARY:,} (expect 9,327)")

# --- 2019 HbA1c labs joined to t0 ---
labs_2019 = lab_results[
    (lab_results['date'] >= '2019-01-01') &
    (lab_results['date'] <  '2020-01-01')
].copy()
labs_2019['patient_id'] = labs_2019['patient_id'].astype(str)

labs_t0 = labs_2019.merge(primary_cohort, on='patient_id', how='inner')
labs_t0['days_from_t0'] = (labs_t0['date'] - labs_t0['t0']).dt.days

# --- Patient flags ---
has_baseline_ids = set(
    labs_t0.loc[
        (labs_t0['days_from_t0'] >= -183) &
        (labs_t0['days_from_t0'] <=    0),
        'patient_id'
    ].unique()
)
has_any_2019_ids = set(labs_t0['patient_id'].unique())

all_primary_ids       = set(primary_cohort['patient_id'])
missing_baseline_ids  = all_primary_ids - has_baseline_ids
overlap               = missing_baseline_ids & has_any_2019_ids

n_missing     = len(missing_baseline_ids)
n_with_any    = len(overlap)
n_with_none   = n_missing - n_with_any
pct_with_any  = (n_with_any / n_missing * 100) if n_missing else 0.0
pct_with_none = 100.0 - pct_with_any

print('=' * 60)
print('MISSING BASELINE — TIMING vs. ABSENCE CHECK')
print('=' * 60)
print(f'Missing pre-GLM baseline HbA1c: {n_missing:,} ({n_missing/N_PRIMARY*100:.1f}% of cohort)')
print(f'  - Had ≥1 HbA1c elsewhere in 2019: {n_with_any:,} ({pct_with_any:.1f}%)')
print(f'  - No 2019 HbA1c at all:           {n_with_none:,} ({pct_with_none:.1f}%)')

# --- Bonus: median days from t0 to first POST-GLM HbA1c (overlap group) ---
overlap_post = labs_t0[
    labs_t0['patient_id'].isin(overlap) &
    (labs_t0['days_from_t0'] > 0)
].copy()

first_post = (
    overlap_post.sort_values('days_from_t0')
                .groupby('patient_id', as_index=False)['days_from_t0']
                .first()
)
median_lag = first_post['days_from_t0'].median()
print(f'\nMedian days from first GLM to first post-GLM HbA1c (overlap): {median_lag:.0f} days')

# --- Response-letter template ---
print()
print('Response-letter sentence:')
print(f'  "Of the {n_missing:,} patients ({n_missing/N_PRIMARY*100:.1f}%) without a qualifying')
print(f'   pre-treatment HbA1c, {n_with_any:,} ({pct_with_any:.1f}%) had an HbA1c recorded')
print(f'   later in 2019 (median {median_lag:.0f} days after first GLM), indicating the')
print(f'   gap reflects measurement timing relative to treatment initiation rather')
print(f'   than absence of glycemic monitoring."')

In [ ]:
# =========================
# 6c) Missing-baseline diagnostic check — BMI parallel
# =========================
# Same logic as 6b, but for BMI (BMI_vital_signs / value column).
# Should reproduce eTable 4's missing flag for BMI (expect 4,825 missing).

# Reuse primary_cohort and N_PRIMARY from 6b. If running 6c standalone,
# uncomment the block below.
# dropout_ids = set()
# for _, id_list in early_dropout_patients.items():
#     dropout_ids.update([str(x) for x in id_list])
# pi = patient_index.copy()
# pi['patient_id'] = pi['patient_id'].astype(str)
# primary_cohort = pi[~pi['patient_id'].isin(dropout_ids)].copy()
# N_PRIMARY = len(primary_cohort)

# --- 2019 BMI records joined to t0 ---
bmi_2019 = BMI_vital_signs[
    (BMI_vital_signs['date'] >= '2019-01-01') &
    (BMI_vital_signs['date'] <  '2020-01-01')
].copy()
bmi_2019['patient_id'] = bmi_2019['patient_id'].astype(str)

bmi_t0 = bmi_2019.merge(primary_cohort, on='patient_id', how='inner')
bmi_t0['days_from_t0'] = (bmi_t0['date'] - bmi_t0['t0']).dt.days

# --- Patient flags ---
has_baseline_bmi_ids = set(
    bmi_t0.loc[
        (bmi_t0['days_from_t0'] >= -183) &
        (bmi_t0['days_from_t0'] <=    0),
        'patient_id'
    ].unique()
)
has_any_2019_bmi_ids = set(bmi_t0['patient_id'].unique())

missing_baseline_bmi_ids = set(primary_cohort['patient_id']) - has_baseline_bmi_ids
overlap_bmi              = missing_baseline_bmi_ids & has_any_2019_bmi_ids

n_missing_b   = len(missing_baseline_bmi_ids)
n_with_any_b  = len(overlap_bmi)
n_with_none_b = n_missing_b - n_with_any_b
pct_with_any_b  = (n_with_any_b / n_missing_b * 100) if n_missing_b else 0.0
pct_with_none_b = 100.0 - pct_with_any_b

print('=' * 60)
print('MISSING BASELINE — BMI: TIMING vs. ABSENCE CHECK')
print('=' * 60)
print(f'Missing pre-GLM baseline BMI: {n_missing_b:,} ({n_missing_b/N_PRIMARY*100:.1f}% of cohort)')
print(f'  - Had ≥1 BMI elsewhere in 2019: {n_with_any_b:,} ({pct_with_any_b:.1f}%)')
print(f'  - No 2019 BMI at all:           {n_with_none_b:,} ({pct_with_none_b:.1f}%)')

# --- Bonus: median days from t0 to first POST-GLM BMI (overlap group) ---
overlap_post_b = bmi_t0[
    bmi_t0['patient_id'].isin(overlap_bmi) &
    (bmi_t0['days_from_t0'] > 0)
].copy()

first_post_b = (
    overlap_post_b.sort_values('days_from_t0')
                  .groupby('patient_id', as_index=False)['days_from_t0']
                  .first()
)
median_lag_b = first_post_b['days_from_t0'].median()
print(f'\nMedian days from first GLM to first post-GLM BMI (overlap): {median_lag_b:.0f} days')

## Section 7 — Export to Excel

In [ ]:
OUT_XLSX = '/content/Revised_Table1.xlsx'

with pd.ExcelWriter(OUT_XLSX, engine='xlsxwriter') as writer:
    sheet = 'Cohort Summary'
    startrow = 2

    summary_df.to_excel(writer, sheet_name=sheet, index=False, startrow=startrow)

    wb = writer.book
    ws = writer.sheets[sheet]

    title_fmt  = wb.add_format({'bold': True, 'font_size': 14, 'align': 'left'})
    header_fmt = wb.add_format({'bold': True, 'font_size': 12, 'bottom': 1})
    body_fmt   = wb.add_format({'font_size': 11})
    bold_row   = wb.add_format({'font_size': 11, 'bold': True})

    ws.merge_range(0, 0, 0, 1, 'Patient Characteristics — Revised Table 1', title_fmt)

    ws.write(startrow, 0, col_left, header_fmt)
    ws.write(startrow, 1, col_right, header_fmt)

    ws.set_column(0, 0, 45, body_fmt)
    ws.set_column(1, 1, 25, body_fmt)

    categories_to_bold = {
        'Demographics',
        'Clinical Characteristics',
        'Comorbidities before June 1, 2019',
    }
    for i, val in enumerate(summary_df.iloc[:, 0].tolist(), start=1):
        if val in categories_to_bold:
            ws.write(startrow + i, 0, val, bold_row)
            ws.write(startrow + i, 1, '',   bold_row)

print(f'Exported revised Table 1 to: {OUT_XLSX}')

## Section 8 — Notes for Table 1 Footnote & Methods Text

**Suggested Table 1 footnote (revised):**

> Values are summarized as N (%). Baseline glycated hemoglobin (HbA1c) and body mass index (BMI) represent the closest recorded value at or before each patient's first glucose-lowering medication (GLM) prescription within 2019. Patients without a qualifying pre-treatment value are reported as missing. Comorbidities indicate documented heart failure (HF) or chronic kidney disease (CKD) between January 1 and June 1, 2019.

**Key numbers for response letter:**
- Original Table 1 reported 14.0% (1,303/9,327) with first HbA1c < 6.4%
- 59.9% of those patients (780/1,303) had their first HbA1c recorded *after* GLM initiation
- Revised Table 1 restricts to pre-treatment values, producing [X]% missing and a reduced <6.4% count of [Y]
- The shift confirms that many original "low" values reflected post-treatment measurements, not diagnostic uncertainty